In [ ]:
%pip install -q PyPDF2
import os
import importlib
import io

genai = importlib.import_module("google.genai")
types = importlib.import_module("google.genai.types")
PyPDF2 = importlib.import_module("PyPDF2")

Note: you may need to restart the kernel to use updated packages.


In [ ]:
API_KEY = os.getenv("GOOGLE_API_KEY")

if not API_KEY:
    raise ValueError("Set the GOOGLE_API_KEY environment variable.")

client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash-lite"

In [4]:
LOCAL_PDF_PATH = "Consumer_Protection_EN.pdf"


def load_pdf_from_local(path):
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return ""

    with open(path, "rb") as f:
        pdf_bytes = f.read()

    pdf_reader = PyPDF2.PdfReader(io.BytesIO(pdf_bytes))
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text() + "\n"
    return text

print("Loading SAMA document from local file...")
full_text = load_pdf_from_local(LOCAL_PDF_PATH)
print(f"Loaded {len(full_text)} characters from the document.")

Loading SAMA document from local file...
Loaded 31817 characters from the document.


In [5]:
def chunk_text(text, chunk_size=1000, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks
 
chunks = chunk_text(full_text)
print(f"✓ Created {len(chunks)} chunks.\n")

✓ Created 38 chunks.



In [6]:
# Expand common abbreviations and synonyms so retrieval doesn't fail
SYNONYM_MAP = {
    "sama": ["saudi arabian monetary authority", "saudi central bank", "central bank", "sama"],
    "samabot": ["sama", "bot", "assistant"],
    "bank": ["bank", "banking", "financial institution", "finance"],
    "complaint": ["complaint", "complain", "grievance", "issue", "problem"],
    "rights": ["rights", "protection", "consumer", "principles"],
    "card": ["card", "credit card", "debit card", "payment card"],
    "transfer": ["transfer", "remittance", "wire", "send money"],
    "loan": ["loan", "finance", "financing", "credit", "lending"],
    "fee": ["fee", "fees", "charge", "charges", "cost"],
    "account": ["account", "accounts", "banking account"],
    "definition": ["definition", "means", "defined as", "refers to", "meaning"],
    "what is": ["means", "defined as", "refers to", "is defined", "definition"],
}
 
def expand_query(question):
    """Expand query with synonyms for better keyword matching."""
    q_lower = question.lower()
    expanded_words = set(q_lower.split())
    for key, synonyms in SYNONYM_MAP.items():
        if key in q_lower:
            expanded_words.update(synonyms)
    return expanded_words
 
def retrieve_context(question, chunks, top_k=5):
    """Improved retrieval using expanded synonyms and bigram matching."""
    query_words = expand_query(question)
    
    # Also add raw question words
    query_words.update(question.lower().split())
    
    # Remove very common stop words
    stop_words = {"the", "a", "an", "is", "are", "was", "were", "in", "of",
                  "to", "and", "or", "for", "with", "what", "how", "why",
                  "does", "do", "i", "my", "me", "can", "will", "be"}
    query_words -= stop_words
 
    scores = []
    for i, chunk in enumerate(chunks):
        chunk_lower = chunk.lower()
        chunk_words = set(chunk_lower.split()) - stop_words
        
        # Word overlap score
        overlap = len(query_words & chunk_words)
        
        # Bonus: exact phrase match
        bonus = 0
        q_stripped = question.lower().strip("?").strip()
        if q_stripped in chunk_lower:
            bonus += 10
        # Bonus for partial phrases (2+ words together)
        words_list = [w for w in question.lower().split() if w not in stop_words]
        for j in range(len(words_list) - 1):
            bigram = words_list[j] + " " + words_list[j+1]
            if bigram in chunk_lower:
                bonus += 3
 
        scores.append((overlap + bonus, i))
 
    scores.sort(reverse=True)
    top_chunks = [chunks[i] for _, i in scores[:top_k]]
    return "\n\n---\n\n".join(top_chunks)

In [7]:
# ── System Prompt ──────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are SAMABot, a professional bilingual (Arabic/English) assistant \
specialized in Saudi financial consumer protection rights. You answer questions \
based on the official SAMA (Saudi Central Bank — Saudi Arabian Monetary Authority) \
Financial Consumer Protection Principles and Rules document.
 
SAMA stands for Saudi Arabian Monetary Authority, also known as the Saudi Central Bank. \
It is the central bank of Saudi Arabia, responsible for supervising and regulating \
financial institutions and protecting financial consumers in the Kingdom.
 
IMPORTANT RULES:
1. Use the DOCUMENT CONTEXT below as your primary source. If the answer is clearly \
   there, quote it accurately and mention the rule number if available.
2. For general factual questions about SAMA (e.g. what is SAMA, who is SAMA) that any \
   person would know, you may answer from general knowledge — but keep it brief and \
   always connect back to consumer protection.
3. If a specific question is genuinely not answered in the document AND is not general \
   knowledge, say: "This specific detail is not covered in the SAMA Consumer Protection \
   document. For further help, contact SAMA directly at 800-125-6666 or visit sama.gov.sa"
4. If the user writes in Arabic, respond fully in Arabic. If English, respond in English.
5. Never give personal legal advice. Explain what the rules state, not what the user \
   should do legally.
6. Be concise, professional, and warm. Use clear simple language.
7. Do not repeat the question back to the user.
 
DOCUMENT CONTEXT (from official SAMA Consumer Protection document):
{context}
 
CONVERSATION HISTORY:
{history}
 
USER QUESTION: {question}
 
Answer:"""
 
# ── RAG Response ───────────────────────────────────────────────────────────────
def get_rag_response(question, chunks, history, temperature=0.3):
    context = retrieve_context(question, chunks)
    history_text = "\n".join(
        [f"User: {q}\nSAMABot: {a}" for q, a in history[-4:]]
    )
 
    prompt = SYSTEM_PROMPT.format(
        context=context,
        history=history_text,
        question=question
    )
 
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=temperature,
                max_output_tokens=400,
                top_p=0.9,
            ),
        )
        return response.text.strip()
    except Exception as e:
        return f"Sorry, an error occurred: {e}"

In [8]:
def run_samabot():
    print("=" * 55)
    print("  SAMABot — Saudi Financial Consumer Protection")
    print("  Powered by SAMA Official Documents")
    print("  Type 'exit' to quit | اكتب 'exit' للخروج")
    print("=" * 55)
    print("\nHello! I am SAMABot. Ask me anything about your rights")
    print("as a financial consumer in Saudi Arabia.\n")
 
    history = []
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye! مع السلامة")
            break
 
        if user_input.lower() in ("exit", "quit", "خروج"):
            print("\nThank you for using SAMABot. مع السلامة! 👋")
            break
 
        if not user_input:
            continue
 
        print("\nSAMABot: ", end="", flush=True)
        response = get_rag_response(user_input, chunks, history)
        print(response)
        print()
        history.append((user_input, response))
 
if __name__ == "__main__":
    run_samabot()

  SAMABot — Saudi Financial Consumer Protection
  Powered by SAMA Official Documents
  Type 'exit' to quit | اكتب 'exit' للخروج

Hello! I am SAMABot. Ask me anything about your rights
as a financial consumer in Saudi Arabia.


SAMABot: SAMA, also known as the Saudi Central Bank, is the central bank of Saudi Arabia. It is responsible for supervising and regulating financial institutions and protecting financial consumers in the Kingdom.


SAMABot: لا يمكن للمستهلكين الاقتراض مباشرة من ساما (البنك المركزي السعودي). ساما هي الجهة التنظيمية والإشرافية للمؤسسات المالية في المملكة العربية السعودية، وهي مسؤولة عن حماية حقوق المستهلكين الماليين. إذا كنت تبحث عن خيارات الاقتراض، فيجب عليك التواصل مع البنوك والمؤسسات المالية المرخصة من قبل ساما.


SAMABot: أهداف ساما (البنك المركزي السعودي) تتمثل في الإشراف على المؤسسات المالية وتنظيمها، وحماية المستهلكين الماليين في المملكة العربية السعودية.


Thank you for using SAMABot. مع السلامة! 👋
